<a href="https://colab.research.google.com/github/2catch2/Hallucination-Audit-engine-/blob/main/HallucinationAuditor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import re
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from enum import Enum

PI  = np.pi
PHI = (1 + 5**0.5) / 2
GATE_THRESHOLD = 1 / 3   # The irreducible remainder principle

class AuditPhase(Enum):
    NOMINAL       = "NOMINAL"
    DEGRADED      = "DEGRADED"
    CRITICAL      = "CRITICAL"
    CIRCUIT_BREAK = "CIRCUIT_BREAK" # Instantly tripped by What-Why null intersections
    HALLUCINATION = "HALLUCINATION" # Shortfall diverging
    VERIFIED      = "VERIFIED"      # Shortfall contracting

@dataclass
class EvidenceQuantum:
    """Atomic unit of grounding evidence."""
    claim_fragment: str
    support_score:  float    # 0.0 to 1.0
    source:         str
    confidence:     float    # Credibility weighting

@dataclass
class GroundingSignal:
    """Measures the instantaneous shortfall (gap) between assertion and grounding."""
    assertion_confidence: float
    grounded_support:     float
    shortfall:            float

    @classmethod
    def measure(cls, assertion_confidence: float, grounded_support: float) -> "GroundingSignal":
        shortfall = max(0.0, assertion_confidence - grounded_support)
        return cls(assertion_confidence, grounded_support, shortfall)

class WhatWhySieve:
    """
    Implements the 1.0 / 3.0 What-Why State Machine logic proof.
    Verifies if the causal logic (Why) constrains and predicts progressive data (What).
    A falsehood/hallucination cannot sustain parallel causal scaling.
    """
    def evaluate_progressive_step(self, claim: str, step_n: int, consistency: float) -> float:
        words = claim.lower().split()
        if len(words) == 0:
            return 0.0

        # Simulate state evaluation step-by-step (3-to-5 step window scaling)
        # High consistency and balanced syntax reduce progressive entropy
        step_factor = min(1.0, (step_n + 1) / 4.0)
        progressive_entropy = np.sin(step_factor * PI) * (1.0 - consistency)

        # Returns the progressive intersection score f(W_n, Y_n)
        return max(0.0, 1.0 - progressive_entropy)

class FalsehoodGate:
    """Fires when shortfall gaps breach the irreducible 1/3 threshold."""
    THRESHOLD = GATE_THRESHOLD

    def __init__(self):
        self.fire_history: List[bool] = []

    def evaluate(self, signal: GroundingSignal) -> bool:
        fired = signal.shortfall > self.THRESHOLD
        self.fire_history.append(fired)
        return fired

    def fire_rate(self) -> float:
        return sum(self.fire_history) / len(self.fire_history) if self.fire_history else 0.0

@dataclass
class HeisenbergLedger:
    """
    Self-referential entropic ledger.
    The act of forced observation/auditing costs energy and disturbs certainty.
    """
    balance: float = 1.0
    observation_cost: float = 0.008 # Quantum overhead per pass

    def observe(self, assertion_rigidity: float) -> float:
        # Forcing absolute certainty on ungrounded claims causes exponential entropic drain
        drain = self.observation_cost * (1.0 + assertion_rigidity)
        self.balance = max(0.0, self.balance - drain)
        return self.balance

class HallucinationAuditEngineV2:
    def __init__(self, max_passes: int = 10):
        self.max_passes = max_passes

    def _extract_linguistic_rigidity(self, claim: str) -> tuple:
        """Measures how static/absolute the claim asserts itself."""
        words = claim.lower().split()
        absolute_markers = {"always", "never", "definitely", "proven", "fact", "undeniably", "certain"}
        hedge_markers = {"might", "could", "possibly", "perhaps", "maybe", "typically", "likely"}

        abs_count = sum(1 for w in words if w in absolute_markers)
        hedge_count = sum(1 for w in words if w in hedge_markers)

        # Rigidity scales up with unhedged absolute terminology
        rigidity = 0.5 + (abs_count * 0.15) - (hedge_count * 0.1)
        base_assertion = 0.6 + (abs_count * 0.08) - (hedge_count * 0.06)

        return min(1.0, max(0.1, rigidity)), min(1.0, max(0.1, base_assertion))

    def audit(self, claim: str, evidence: Optional[List[EvidenceQuantum]] = None) -> Dict[str, Any]:
        sieve = WhatWhySieve()
        gate = FalsehoodGate()
        ledger = HeisenbergLedger()

        # Cumulative Phi-scaled evidence parsing
        grounding_score = 0.0
        if evidence:
            for idx, eq in enumerate(evidence):
                phi_decay = PHI ** (-(idx + 1)) # Diminishing returns of stacked proofs
                grounding_score += eq.support_score * eq.confidence * phi_decay
            grounding_score = min(1.0, grounding_score / PHI)

        rigidity, base_assertion = self._extract_linguistic_rigidity(claim)

        shortfall_history = []
        lambda_history = []
        phase = AuditPhase.NOMINAL
        circuit_broken = False

        # Multi-pass dynamic contraction audit loop
        for pass_n in range(self.max_passes):
            # 1. Deduct quantum observation cost
            remaining_energy = ledger.observe(rigidity)

            # 2. π-Cycle modulated consistency
            cycle_modulation = np.cos(2 * PI * pass_n / self.max_passes)
            current_consistency = max(0.1, min(1.0, (1.0 - rigidity * 0.3) + (cycle_modulation * 0.1)))

            # 3. Progressive What-Why Sieve Step
            intersection_score = sieve.evaluate_progressive_step(claim, pass_n, current_consistency)

            # CRITICAL HIT: What-Why Null Intersection Circuit Breaker Rule
            if intersection_score < 0.35:
                phase = AuditPhase.CIRCUIT_BREAK
                circuit_broken = True
                break

            # 4. Measure dynamic Grounding Signal Shortfall
            assertion_confidence = base_assertion * current_consistency * remaining_energy
            signal = GroundingSignal.measure(assertion_confidence, grounding_score)
            shortfall_history.append(signal.shortfall)

            # 5. Evaluate the 1/3 Falsehood Gate
            gate.evaluate(signal)

            # 6. Calculate Lambda (λ Trajectory)
            if pass_n > 0 and shortfall_history[0] > 1e-6:
                current_lambda = signal.shortfall / shortfall_history[0]
                lambda_history.append(current_lambda)
            else:
                lambda_history.append(1.0)

            # Instant Early-Closure checks
            if len(lambda_history) >= 3:
                # Contracting trajectory -> Confirmed truth
                if lambda_history[-1] < 0.88:
                    phase = AuditPhase.VERIFIED
                    break
                # Diverging trajectory -> Unmoored hallucination
                elif lambda_history[-1] > 1.08 or remaining_energy < 0.1:
                    phase = AuditPhase.HALLUCINATION
                    break

        # Final fallback evaluations if the audit finishes all loops without early closure
        if phase not in [AuditPhase.VERIFIED, AuditPhase.HALLUCINATION, AuditPhase.CIRCUIT_BREAK]:
            final_lambda = lambda_history[-1] if lambda_history else 1.0
            if final_lambda < 1.0:
                phase = AuditPhase.VERIFIED
            else:
                phase = AuditPhase.HALLUCINATION

        mean_lambda = float(np.mean(lambda_history)) if lambda_history else 1.0
        final_shortfall = shortfall_history[-1] if shortfall_history else 0.0

        return {
            "claim": claim,
            "verdict": phase.value,
            "final_shortfall": round(final_shortfall, 4),
            "lambda_trajectory": round(mean_lambda, 4),
            "gate_fire_rate": f"{gate.fire_rate() * 100:.1f}%",
            "remaining_ledger_energy": round(ledger.balance, 4),
            "circuit_broken": circuit_broken
        }

In [2]:

# Initialize Engine
engine = HallucinationAuditEngineV2()

# TEST CASE A: Dynamic Grounding (Contracts gracefully)
claim_a = "The golden ratio φ equals approximately 1.618 and reliably scales growth systems."
evidence_a = [
    EvidenceQuantum("φ math constant", 0.98, "geometry", 0.95),
    EvidenceQuantum("logarithmic system scaling", 0.88, "system dynamics", 0.90)
]

# TEST CASE B: Unmoored Absolute Assertion (Diverges & Triggers the What-Why Break)
claim_b = "Einstein definitely proved that consciousness creates physical realities instantly."
evidence_b = None # Completely ungrounded assertion